# 第2回演习：使用低秩适应（LoRA）学习回答格式，比较训练前后

以虚构课程的常见问题解答（FAQ）为题材，对小型指令微调模型进行短时间的低秩适应（Low-Rank Adaptation，LoRA）训练，比较训练前后**哪些发生变化、哪些没有变化**。评价分为**格式遵守率**（是否遵守包含 `answer` 和 `needs_review` 的 JSON 格式）和**内容正确率**。

**本演习要确认的内容**

1. 即使在提示词中指定格式，小模型也可能无法遵守（第3节，训练前）。
2. 使用 LoRA 训练几分钟后，模型大多能够遵守格式（第5节，训练后）。
3. 即使格式变整齐，也不能保证模型能正确判断 FAQ 未记载的问题：是请求确认，还是直接断言（第5、6节）。
4. 拓展：将同一模型以4位量化加载，即使不训练，输出也可能改变（第8节）。

**流程**

| 章节 | 内容 | 时间参考（T4） |
| --- | --- | --- |
| 0～3 | 环境、配置、生成数据、加载模型、训练前评价 | 约5分钟 |
| 4～7 | 训练、训练后评价、知识问题、比较与提交 | 约10分钟（其中训练4～5分钟） |
| 8（可选） | 以4位量化加载并比较 | 约2分钟 |

课前可以先运行第0～3节，确认环境可用，减少上课当天的问题。Colab 会话断开后变量会消失，因此需要时请从第0节重新执行。

**使用环境**：Google Colab（T4 图形处理器，GPU；免费额度）。模型为 `Qwen/Qwen3.5-0.8B`（Apache-2.0）。数据由本笔记本生成，均为虚构内容，不包含真实截止日期或网址。

**阅读方法**：各节说明包括“操作”“观察重点”和“遇到问题时”。请按顺序从上到下运行单元格；跳过单元格可能导致后续变量不存在。

版本：1.1（2026-10-18）

**双实验版**：先完整运行中文 FAQ 的第0～8节，再重新加载基础模型执行中文词汇表实验；第8节为可选观察，默认跳过。原始模型输出不继承。

## 0. 检查环境并安装软件库

**操作**：确认已分配 GPU，并安装已验证的固定版本软件库。

**观察重点**

- 第一个单元格显示 `Tesla T4`。
- 第二个单元格末尾显示版本列表，例如 `transformers: 5.18.0`。安装约需1分钟。

**遇到问题时**

- 出现“未找到 GPU”：在菜单中选择“运行时”→“更改运行时类型”→“T4 GPU”，然后重新执行。资源繁忙无法分配时，稍后再试。
- 与 `cudf`、`pyarrow` 有关的红色 `ERROR: pip's dependency resolver ...` 是 Colab 预装软件包之间的依赖冲突警告；原演习说明称其不影响本实验，可以继续。

In [ ]:
# 【中文语法说明】subprocess.run(...) 启动外部命令；参数列表中的每个字符串是一个独立参数。
# try / except 用于在命令不存在时提供备用提示，避免程序直接终止。
# .stdout 是命令的标准输出；or 在左侧为空时使用右侧备用文字。
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU运行时没有 nvidia-smi 命令。
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# 【中文语法说明】PINS 是字典：左侧是软件包名称，右侧是固定版本。
# for k, v in PINS.items() 同时取出每对“名称、版本”；f 字符串中的花括号会插入变量。
# %pip 是 Colab 单元格的特殊命令，不是普通 Python 语句；版本不宜随意修改。
# 在Colab T4环境验证的软件版本；torch使用Colab自带版本。
PINS = {
    "transformers": "5.18.0",   # Qwen3.5需要transformers 5系列。
    "peft": "0.21.1",           # LoRA的实现库。
    "trl": "1.14.2",            # 监督微调训练循环。
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13与pyarrow可能出现依赖冲突警告；原课程称本演习不使用cudf。
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 配置

**操作**：把模型名称、训练轮数等集中保存在字典 `CONFIG` 中。第7节会将这些配置写入提交文件，便于之后确认实验条件。

- 固定 `revision` 是为了每次使用相同权重，便于比较训练前后及不同人的结果。
- `num_train_epochs: 20` 表示将14条训练数据重复20轮。每轮2步，共40步；T4训练约4～5分钟。
- `gen: {"do_sample": False}` 表示贪心生成，使训练前后生成条件一致，减少采样波动的影响。

将 `USE_DRIVE` 设为 `True` 会挂载 Google Drive，并把训练后的适配器（adapter）和结果文件保存到云端硬盘，以便会话断开后保留。一次会话内完成时，可以保持 `False`。

**观察重点**：显示的 JSON 配置。

In [ ]:
# 【中文语法说明】CONFIG 是字典，使用 CONFIG["键名"] 读取对应配置。
# Path 表示文件路径；mkdir(parents=True, exist_ok=True) 会按需创建目录，并允许目录已存在。
# if USE_DRIVE: 后缩进的语句只在条件为真时执行；False 和 True 是布尔值。
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小型指令微调模型，无量化时可放入T4。
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # 固定模型版本，以便使用相同权重。
    "seed": 42,
    "num_train_epochs": 20,  # 训练数据14条，每轮2步，20轮共40步。
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA矩阵的秩；越小通常可训练参数越少。
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 评价时固定贪心生成，便于前后比较。
    "work_dir": "/content/qlora_lab",                          # 结果保存路径。
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True时的保存路径。
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # 设为True时挂载云端硬盘并保存适配器与输出。
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 生成虚构 FAQ 和监督训练数据

**操作**：根据5条虚构 FAQ，生成训练集（train）、验证集（valid）和测试集（test）。

模型输入是“FAQ＋问题”；期望输出是一行 JSON：`{"answer": <字符串>, "needs_review": <布尔值>}`。

- FAQ 有依据时，依据 FAQ 回答，`needs_review` 为 `false`。
- FAQ 没有记载时，回答“この FAQ には記載がありません。運営に確認が必要です。”（此 FAQ 没有记载，需要向运营方确认），`needs_review` 为 `true`。

**数据划分**

- 将同一 FAQ 条目的不同问法按**模板**分配到 train / valid / test，避免训练中见过的原句直接出现在测试中。
- test 有21题（有依据15题、无依据6题），其中另行准备了只用于 test 的问题。
- 标准答案保存在 `gold` 中，包括 `needs_review` 的值和回答必须包含的词句；不传给模型。
- train 有14条（有依据10条、无依据4条），valid 有7条。FAQ 只有5项，数据量很少，同一例子会重复训练。重复次数由第4节的 `num_train_epochs` 决定，而不是复制数据；数据文件保存实际样本，训练量由配置控制。

**观察重点**：显示 `train=14 valid=7 test=21 gold=21` 和一条训练示例。如果同一个问题在 train / valid / test 中重复，会在这里报错。

In [ ]:
# 【中文语法说明】花括号表示字典；FAQ.items() 每次取出一个键和值。
# 列表中的四个问法按前两个、第三个、第四个分别用于训练、验证、测试。
# 列表切片 a[:2] 表示从开头取到下标2之前；a[2:3] 只取下标2。
# for ... in ... 依次处理数据；列表推导式 [表达式 for 变量 in 数据] 是批量创建列表的简写。
# assert 用于检查条件；len(set(问题)) 与总数相等时，表示没有完全相同的重复问题。
# json.dumps 将 Python 对象转成 JSON 文字；ensure_ascii=False 允许中文直接显示。
FAQ = {"deadline":"作业截止时间是9月30日23:59（日本时间）。", "destination":"提交到学习平台的作业栏目。", "format":"提交一个PDF文件，文件名为学号_作业编号.pdf。", "office_hours":"每周二17:00至18:00接受提问。", "late":"截止后48小时内可迟交，但会扣分。"}
FAQ_TEXT="\n".join(f"- {v}" for v in FAQ.values())
# 五组问题：前两条训练，第三条验证，第四条测试；词句用于评分。
GROUNDED={
 "deadline":(["作业什么时候截止？","提交期限是什么时候？","最后提交时间？","截止的日期和时间？"],["9月30日","23:59"]),
 "destination":(["作业提交到哪里？","在哪里交作业？","上传到哪个栏目？","提交入口在哪里？"],["学习平台","作业"]),
 "format":(["提交格式是什么？","需要提交什么文件？","提交几份PDF？","文件格式有什么要求？"],["PDF"]),
 "office_hours":(["什么时候可以提问？","答疑时间是什么？","周几接受提问？","答疑是几点到几点？"],["周二","17:00"]),
 "late":(["迟交有什么规定？","逾期还能交吗？","迟交会扣分吗？","截止后提交如何处理？"],["48小时","扣分"])}
UNGROUNDED=["允许重新提交吗？","缺课怎么处理？","可以看到他人成绩吗？","成绩何时公布？","截止日期能延长吗？","可以小组提交吗？","作业占多少分？","课程录像会公开吗？"]
TEST_EXTRA_GROUNDED={"deadline":["作业最迟何时提交？","截止时刻是几点？"],"destination":["向哪个平台提交？","作业需要上传到哪？"],"format":["必须提交什么文件格式？","文件应使用什么格式？"],"office_hours":["每周答疑安排是什么？","接受问题的时段是哪天？"],"late":["超过期限还可以提交吗？","作业晚交如何处理？"]}
TEST_EXTRA_UNGROUNDED=["作业有字数限制吗？","提交后会收到反馈吗？","参考文献格式有规定吗？","答疑需要预约吗？"]
UNGROUNDED_ANSWER="资料中没有记载，需要确认。"
def make_example(question,answer,needs_review):
    """打包问题、资料与标准答案；模型评价时不会读取target。"""
    return {"question":question,"faq":FAQ_TEXT,"target":json.dumps({"answer":answer,"needs_review":needs_review},ensure_ascii=False),"needs_review":needs_review}
def split_templates(templates):
    """按不同问法划分训练、验证与测试。"""
    return templates[:2],templates[2:3],templates[3:4]
train,valid,test,gold=[],[],[],{}
for key,(questions,words) in GROUNDED.items():
    tr,va,te=split_templates(questions)
    for q in tr: train.append(make_example(q,FAQ[key],False))
    for q in va: valid.append(make_example(q,FAQ[key],False))
    for q in te+TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q,FAQ[key],False));gold[q]={"needs_review":False,"must_include":words}
for i,q in enumerate(UNGROUNDED):
    ex=make_example(q,UNGROUNDED_ANSWER,True);bucket=[train,train,valid,test][i%4];bucket.append(ex)
    if bucket is test:gold[q]={"needs_review":True,"must_include":["确认"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q,UNGROUNDED_ANSWER,True));gold[q]={"needs_review":True,"must_include":["确认"]}
all_questions=[e['question'] for e in train+valid+test]
assert len(set(all_questions))==len(all_questions)
print(f"训练={len(train)} 验证={len(valid)} 测试={len(test)} 标准答案={len(gold)}")
for name,rows in (("train",train),("test",test)):
    (WORK/f"{name}.jsonl").write_text("\n".join(json.dumps(e,ensure_ascii=False) for e in rows),encoding="utf-8")
(WORK/"gold.json").write_text(json.dumps(gold,ensure_ascii=False,indent=2),encoding="utf-8")

## 3. 加载模型并进行训练前评价

**操作**：不使用量化，以32位精度加载基础模型，对 test 的21题生成回答并评分。

提示词 `SYSTEM` 已经要求模型“必须以一行 JSON 返回 `answer` 和 `needs_review`”。也就是说，**已经给出格式指令**；训练前要检查模型能否遵守。

**评分方式**（第5节使用相同函数）

- **格式**：能解析为 JSON，恰好包含 `answer` 和 `needs_review` 两个键，类型分别为字符串和布尔值。
- **内容**：`needs_review` 与标准答案一致，且 `answer` 包含规定的关键词，例如“9 月 30 日”和“23:59”。

内容评分需要 `needs_review` 一致。因此，缺少该键时，即使文字正确，内容也判为不通过。请自行阅读表格中的回答原文。

`Qwen/Qwen3.5-0.8B` 也能处理图像，但通过 `AutoModelForCausalLM` 加载时只使用文本部分。

**观察重点**

- 第一个单元格可能显示 `重みのメモリ(GB): 3.01`，意为权重占用约3.01 GB。下载模型约1.75 GB，约需1分钟。
- 第二个单元格显示一行汇总（`format_rate`、`content_rate` 等）及21题的输出表；评价约需1分钟。训练前许多输出可能缺少 `needs_review`。

**原演习说明中可忽略的提示**

- `You are sending unauthenticated requests to the HF Hub`：未登录 Hugging Face 下载模型的提示。公开模型可不登录下载；需要登录的情况包括受限或私有模型、上传内容，以及请求达到未登录额度。原资料引用的未登录上限为每个 IP 每5分钟3000次文件获取（2025年9月时点；[Hub 限流说明](https://huggingface.co/docs/hub/rate-limits)）。本笔记本所用模型公开且不要求同意条款；如需登录，可在 Colab 左侧密钥图标中登记名为 `HF_TOKEN` 的秘密，并允许本笔记本访问。
- `... is falling back to its reference PyTorch implementation ...`：部分线性注意力层未安装高速实现，程序会使用较慢的参考实现，提示约出现4行。

**此处检查**：阅读缺失的键和回答与 FAQ 的一致性。第5节会与训练后输出对照。原文也保存在 `/content/qlora_lab/eval_before.json`。

In [ ]:
# 【中文语法说明】from ... import ... 从软件库引入功能；模型的设备优先选 CUDA，其次 MPS，最后 CPU。
# 变量 = A if 条件 else B 是条件表达式：条件成立选择 A，否则选择 B。
# def 定义函数；函数参数 example 是一条样本，return 将结果交回调用者。
# with torch.no_grad() 表示这里仅生成回答，不记录训练梯度，通常能节省内存。
# **inputs 会将字典中的键值拆成具名参数传给函数；它不是乘方。
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 不量化；显式指定权重以32位精度加载。
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 训练前检查模型能否遵守提示词中的格式要求。
SYSTEM = '你是课程FAQ问答助手。请只依据FAQ，用中文输出一行JSON，包含字符串answer和布尔值needs_review；资料没有记载时回答“需要确认”并设为true。'

def build_messages(example):
    """将FAQ和问题组成模型输入，评价时不发送标准答案。"""
    user = f"FAQ:\n{example['faq']}\n\n问题: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    """使用指定模型生成回答，不更新模型参数。"""
    # 关闭思考块，仅生成回答，并保持评价条件一致。
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# 【中文语法说明】try / except 捕获 JSON 解析失败；isinstance 用于检查对象类型。
# set(obj) 是字典键的集合；集合减法用于检查有没有额外的键。
# obj.get("键", 默认值) 在键不存在时返回默认值，避免直接索引报错。
# all(...) 要求括号中的检查全部为真；评分依赖关键词，不等于完整的语义正确性判断。
# 列表推导式逐题生成表格行；sum(布尔值序列) 会把 True 计为1、False计为0。
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 显示完整输出文字，不截断列宽。

def check_format(text: str):
    '''检查JSON键和类型，并返回失败原因。'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''对照确认标记和关键词评分；语义仍需人工复核。'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    """将布尔评分转换为表格标记。"""
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''逐题评价并保存原文和指标。'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 格式不合格时，尝试提取JSON部分单独检查内容。
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 对照标准标记，检查输出中缺失的键及FAQ内容。
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA 训练（T4约4～5分钟）

### LoRA 与 QLoRA 的区别

- **LoRA**：固定基础模型权重，只训练加在部分层上的小矩阵，即适配器（adapter）。
- **QLoRA**：将基础权重量化为4位保存，再训练适配器；适配器本身不量化。

QLoRA 的作用是减少基础权重的存储。相较16位，4位权重理论存储约为四分之一；70亿参数的权重约从14 GB降到3.5 GB，不含其他开销。

| 方面 | LoRA（不量化） | QLoRA（4位） |
| --- | --- | --- |
| 权重内存 | 较大 | 较小，约为16位的1/4 |
| 基础模型输出 | 保持原精度 | 可能受量化误差影响，程度因模型而异 |
| 训练速度 | 基准 | 不一定更快，量化权重转换可能使其变慢 |
| 依赖 | peft | peft 与 bitsandbytes，需要 CUDA GPU |

**实现 QLoRA 时的注意事项**

- 加载时传入 `BitsAndBytesConfig(load_in_4bit=True, ...)`，添加 LoRA 前调用 `prepare_model_for_kbit_training(model)`。
- 存储精度（4位）与计算精度（`bnb_4bit_compute_dtype`，例如16位）是不同设置。
- 训练前后要使用相同的量化条件，否则会混淆量化与训练的影响。
- 保存或复用适配器时，同时记录基础模型 ID、revision 和量化设置。
- 有些模型不建议以4位训练。[相关训练库指南](https://unsloth.ai/docs/models/qwen3.5/fine-tune)指出 Qwen3.5 的量化误差可能较大，因此不推荐4位训练。

本实验的模型不量化也能放入 T4，因此使用 LoRA。第8节可观察4位加载后的内存和输出变化。

### 执行训练

**操作**：把14条训练数据转换成对话格式（system / user / assistant），添加 LoRA 后训练20轮、40步；适配器保存在 `adapter/`。

- `target_modules` 指定插入 LoRA 的层，包括注意力投影（例如 `q_proj`）和多层感知机投影（例如 `gate_proj`）。本模型24层中18层使用线性注意力，只有其余6层包含 `q_proj` 等；所有层都有多层感知机。
- 批次4 × 梯度累积2，每步相当于处理最多8条。14条数据每轮2步（8条和6条），20轮共40步，每条样本使用20次。学习率预热和余弦衰减作用于这40步。
- 训练后的模型保存在 `tuned`。由于适配器插入基础模型，训练后即使用 `model` 生成也会包含适配器效果。需要训练前输出时，按第6节暂时禁用适配器。

**观察重点**

- `trainable params: ... || all params: ... || trainable%: ...`：LoRA 只训练全部参数中的一小部分。
- `Training Loss` 表每10步记录一次；原资料参考从约1.5下降到第40步约0.01～0.02。若没有下降，检查配置。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`：原资料给出的 T4 参考时间为240～290秒。步数和轮数用于确认14条数据重复20轮的执行情况。

`The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` 表示结束等词元配置已与分词器对齐。

训练中断或无法训练时，请阅读第4b节。

In [ ]:
# 【中文语法说明】to_chat 将一条样本整理为 messages：输入消息加上 assistant 的期望输出。
# Dataset.from_list 把 Python 列表整理成训练器可读取的数据对象，不会额外下载另一份数据。
# peft_config 中的配置控制低秩适应；trainer.train() 才开始训练。
# model.config.use_cache 控制缓存；tuned.eval() 切换到评价模式，减少训练模式的随机影响。
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    """把输入和标准答案整理成训练器需要的消息格式。"""
    # 训练样本由system、user输入和assistant标准回答组成，使用与评价相同的输入构造函数。
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# 模型部分层使用线性注意力；gate/up/down投影存在于所有层。
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 设为True会启用16位混合精度计算，仅支持CUDA，可能出现nan损失。
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # 显示LoRA可训练参数的比例。
t0 = time.time()
trainer.train()  # 中途停止时的恢复方法见第4b节。
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 切换到评价模式，避免dropout导致额外波动。

### 4b. 训练中断或无法训练时

**中途停止**：检查点每10步保存在 `/content/qlora_lab/ckpt/`。如果文件仍在，重新运行第0～3节，再把训练单元格的 `trainer.train()` 改为 `trainer.train(resume_from_checkpoint=True)` 并执行，即可从最后检查点继续到40步、20轮。如果运行时被删除、文件不存在，则需要重新训练或采用下面的方法。

**无法训练**：例如没有 GPU 或规定时间内无法完成时，可使用讲师在 T4 上运行同一笔记本得到的输出与分数进行比较。第5节起需要训练后的模型 `tuned` 才能运行，因此请对照讲师结果阅读说明。提交时将第7节的 `executed_training_myself` 设为 `False`，并写明自己实际完成的范围。

## 5. 训练后评价（测试题与生成设置和训练前相同）

**操作**：使用训练后的 `tuned`，以完全相同的21题和生成设置进行评价，对照训练前后。改变的是是否使用适配器。

**观察重点**

- 比较表中 `format_rate`（格式遵守率）和 `content_rate`（内容正确率）的变化。
- `退行`：训练前格式和内容都通过、训练后却失败的问题；理想情况下为空列表。
- 阅读每题原文，区分剩余错误：无依据问题套用 FAQ 其他内容并断言；有依据问题却声称没有记载并要求确认。

训练后的回答可能接近训练范例中的 FAQ 原文。格式和措辞变整齐，并不保证模型已经学会判断 FAQ 是否有依据。

即使固定随机种子，GPU 训练也不保证完全复现。比例相近时，具体失败问题也可能不同；请按错误类型比较。

In [ ]:
# 【中文语法说明】zip(before_rows, after_rows) 把训练前后的对应行按位置配对，因此两次评价必须使用相同顺序。
# [b["question"] for b, a in ... if 条件] 会筛选出符合条件的问题。
# f 字符串中的 {表达式} 会计算表达式后插入文本；DataFrame 用于展示表格。
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 找出训练前通过、训练后失败的问题。
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 逐题检查输出原文及格式、内容评分。
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 考查知识的问题（学会格式不保证知识正确）

**操作**：在格式实验之外，询问3个 FAQ 和训练数据都没有涉及的事实，并加入1个 FAQ 内的问题作为对照。对无依据问题，期望回答“需要确认（`needs_review: true`）”。

| 分类 | 含义 |
| --- | --- |
| `format_error` | 不遵守 JSON 格式，训练前较常见 |
| `needs_review` | 回答“没有记载，需要确认”，对无依据问题是期望结果 |
| `asserted` | 以 `needs_review: false` 作出断言；若 FAQ 无依据，则属于无依据断言 |

**观察重点**：训练前可能多为格式错误，训练后可能同时出现请求确认与断言。表格也显示原文，请检查断言时是否挪用了 FAQ 其他条目。不要预先断定“微调绝对不能改善知识”，应报告本次数据下的实际结果。

训练前输出通过临时禁用适配器取得：`tuned.disable_adapter()`。

In [ ]:
# 【中文语法说明】列表中的每个元组保存“问题、是否有依据”；for q, supported 会把两项分别取出。
# with tuned.disable_adapter(): 只在该代码块内暂时禁用适配器，以取得基础模型输出。
# 字典中的布尔值用于记录资料是否支持；不能用问题是否恰好出现在 gold 来代替。
KNOWLEDGE_CASES=[("最终课题的截止日期？",False),("负责人叫什么？",False),("下次课程主题是什么？",False),("作业提交的截止日期和时间？",True)]
def classify(raw):
    """检查格式并分类为需要确认或断言。"""
    ok,_,obj=check_format(raw)
    if not ok or obj is None:return "格式错误"
    return "需要确认" if obj["needs_review"] else "断言"
knowledge_rows=[]
for q,supported in KNOWLEDGE_CASES:
    ex=make_example(q,"",not supported)
    with tuned.disable_adapter():before_raw=generate(tuned,ex)
    after_raw=generate(tuned,ex)
    knowledge_rows.append({"问题":q,"FAQ有依据":supported,"训练前分类":classify(before_raw),"训练前输出":before_raw,"训练后分类":classify(after_raw),"训练后输出":after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. 比较表与提交

**操作**：将配置、软件库版本、训练前后汇总和退化问题写入 `result.json`。

**提交内容**

- `result.json`、`eval_before.json`、`eval_after.json`：位于 `/content/qlora_lab`，可在左侧文件列表右键下载；`USE_DRIVE = True` 时也会复制到 Drive。
- 第5节的比较表及剩余错误的类型说明。
- 第6节的结果：哪些回答变为请求确认，哪些仍然断言。
- 自己实际执行训练的范围；使用讲师结果时，将 `executed_training_myself` 设为 `False`。

In [ ]:
# 【中文语法说明】字典推导式 {k: v for k, v in ... if 条件} 逐项生成新字典，并按条件排除路径。
# 模块对象的 __name__ 是模块名称；json.dumps(..., indent=2) 将结果排版为易读 JSON。
# write_text 写入文件；结果描述的训练状态必须与实际执行情况一致。
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 如果使用讲师结果而非自己训练，应设为False。
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

## 8. 拓展：以4位量化加载，比较内存和输出（可选）

**操作**：以4位（NF4）加载同一模型，观察权重内存和训练前输出的变化。QLoRA 是在4位基础模型上加入 LoRA 进行训练的方法；本节不训练。

- bitsandbytes 需要 CUDA GPU，只能在 GPU 运行时使用，安装约1分钟。
- 4位加载使用 `device_map="auto"` 分配到 GPU，以适配 bitsandbytes 的加载方式。
- 如果尝试对4位模型训练，先执行 `prepare_model_for_kbit_training(model_4bit)`，再按照第4节训练，并确保训练前后都以4位基础模型评价。

**观察重点**

- 原资料参考：32位权重约3 GB，4位不足1 GB。
- 因为没有训练，与32位训练前结果 `before` 的差异来自量化。有些问题可能变好；有些可能把 `needs_review` 输出成字符串 `"false"` 而未通过类型检查，或对无依据问题直接断言“是”“没有”。变化方向并不固定。

In [ ]:
# 【中文语法说明】RUN_OPTIONAL_4BIT 是开关：False 时跳过整个缩进代码块，True 时才安装并加载4位模型。
# subprocess.run([...], check=True) 运行外部安装命令；失败时抛出错误。
# 与32位比较前，需要确认其他条件相同；本节只是加载观察，不进行训练。
RUN_OPTIONAL_4BIT = False  # 可选：改为True才运行原第8节
if RUN_OPTIONAL_4BIT:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes==0.50.*"], check=True)
    from transformers import BitsAndBytesConfig

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                             bnb_4bit_compute_dtype=torch.float16,             # 计算精度为16位；T4不支持bf16。
                             bnb_4bit_use_double_quant=True)
    model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
    print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

    # 此处没有训练，与32位训练前结果的差异用于观察量化影响。
    q_summary, q_rows = evaluate(model_4bit, "before_4bit")
    display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
    display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "4bit・学習前の出力": r["raw"],
                           "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))

## 9. 词汇表独立实验（新增）
前面第0～8节保留原流程。此处先清除FAQ模型，再重新载入原基础模型；词汇表训练、测试和检索结果另存目录。

默认使用内嵌中文数据快照。如需加载你上传的最新文件，请将 glossary_CN.json 与 glossary_sft_CN.jsonl 上传到 Colab 左侧文件区，并把下一数据单元格的 DATA_MODE 改为“上传”；运行时断开后临时文件需要重新上传。


In [ ]:
# 【中文语法说明】globals() 返回当前全局变量的字典；if name in globals() 判断变量是否存在。
# del globals()[name] 删除该变量引用；gc.collect() 请求回收无引用对象，再清理GPU缓存。
# 删除引用不一定保证所有显存立即释放，因此下一实验仍需检查可用显存。
# 保存FAQ的对照结果，释放FAQ实验的模型，再执行词汇表实验。
import gc
faq_original_result = result
for name in ["model", "tuned", "trainer", "model_4bit"]:
    if name in globals(): del globals()[name]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# 【中文语法说明】本节为词汇表实验重新建立配置和辅助函数，与前面的 FAQ 实验使用独立目录。
# **inputs 将输入字典拆开作为参数；with ...: 限定上下文管理器的作用范围。
# answer_fn or 默认函数表示：提供了检索函数时用它，否则让模型生成。
# for k in (...) if k in ex 是字典推导式：只把允许公开的字段传给回答函数，避免泄露答案。
# *objects 接收任意数量的位置参数；这里只用于显式说明清理对象，实际释放仍要删除外部引用。
from pathlib import Path
import json, time, re, gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

CONFIG = {"base_model":"Qwen/Qwen3.5-0.8B", "revision":"2fc06364715b967f1860aea9cf38778875588b17", "seed":42, "epochs_faq":20, "epochs_glossary":3, "lr":2e-4, "lora_r":16, "lora_alpha":32, "max_seq_len":1024, "max_new_tokens":256}
WORK = Path('/content/lecture02_cn_advanced'); WORK.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "未检测到GPU，请在运行时设置中选择T4"
set_seed(CONFIG['seed'])
tok = AutoTokenizer.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'])
if tok.pad_token is None: tok.pad_token = tok.eos_token
SYSTEM = '请依据给出的资料用中文回答。只输出一行JSON，恰好包含字符串answer和布尔值needs_review。资料没有依据时回答“需要确认”，并将needs_review设为true。'
GLOSSARY_SYSTEM = '请根据学到的课程词汇表回答。只输出一行JSON，恰好包含字符串answer和布尔值needs_review。未收录时回答“需要确认”，并将needs_review设为true。'

def load_base():
    """加载干净的基础模型；每个实验分别调用，避免前一次LoRA训练影响后一次。"""
    set_seed(CONFIG['seed'])
    m = AutoModelForCausalLM.from_pretrained(CONFIG['base_model'], revision=CONFIG['revision'], dtype=torch.float32).to('cuda')
    m.config.use_cache = True
    return m.eval()

def messages(ex, include_answer=False):
    """将资料和问题转换为对话消息；训练时附加标准答案，评价时不附加。"""
    system = SYSTEM if ex['source'] == 'faq' else GLOSSARY_SYSTEM
    user = ('资料：' + ex['context'] + '\n问题：' if ex['source'] == 'faq' else '问题：') + ex['question']
    out = [{'role':'system','content':system}, {'role':'user','content':user}]
    if include_answer: out.append({'role':'assistant','content':json.dumps({'answer':ex['answer'],'needs_review':ex['needs_review']},ensure_ascii=False)})
    return out

def generate(model, ex):
    """根据问题生成回答；评价时不向模型提供标准答案。"""
    prompt = tok.apply_chat_template(messages(ex), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad(): result = model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'], do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(result[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def score(raw, reference):
    """检查JSON格式、确认标记和关键词；关键词评分不能代替人工事实检查。"""
    try: obj=json.loads(raw)
    except (ValueError, TypeError): return False, False
    valid=isinstance(obj,dict) and set(obj)=={'answer','needs_review'} and isinstance(obj['answer'],str) and type(obj['needs_review']) is bool
    if not valid: return False, False
    content=obj['needs_review']==reference['needs_review'] and all(k in obj['answer'] for k in reference['keywords'])
    return valid, content

def evaluate(model, tests, label, answer_fn=None):
    """逐题生成或检索，再与独立标准答案比较，保存明细和汇总。"""
    rows=[]; start=time.time()
    for ex in tests:
        public={k:ex[k] for k in ('question','context','source')}
        raw=answer_fn(public) if answer_fn else generate(model, public)
        fmt, content=score(raw, ex)
        rows.append({'类别':ex['group'],'问题':ex['question'],'标准答案':ex['answer'],'需确认标准':ex['needs_review'],'输出':raw,'格式通过':fmt,'内容通过':content})
    summary={'实验':label,'题数':len(rows),'格式遵守率':sum(x['格式通过'] for x in rows)/len(rows),'内容正确率':sum(x['内容通过'] for x in rows)/len(rows),'评价耗时秒':round(time.time()-start,2)}
    (WORK/f'{label}.json').write_text(json.dumps({'summary':summary,'rows':rows},ensure_ascii=False,indent=2),encoding='utf-8')
    display(pd.DataFrame(rows)); return summary

def train_lora(base, train, valid, label, epochs):
    """将训练样本转为对话格式，训练LoRA适配器并返回训练后模型及时间。"""
    train_ds=Dataset.from_list([{'messages':messages(e,True)} for e in train])
    valid_ds=Dataset.from_list([{'messages':messages(e,True)} for e in valid])
    lengths=[len(tok.apply_chat_template(row['messages'],tokenize=True,add_generation_prompt=False,enable_thinking=False)) for row in train_ds]
    assert max(lengths)<=CONFIG['max_seq_len'], '训练数据超长，请检查样本或调整上限'
    base.config.use_cache=False
    lora=LoraConfig(r=CONFIG['lora_r'],lora_alpha=CONFIG['lora_alpha'],lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    args=SFTConfig(output_dir=str(WORK/label/'ckpt'),num_train_epochs=epochs,per_device_train_batch_size=4,gradient_accumulation_steps=2,learning_rate=CONFIG['lr'],lr_scheduler_type='cosine',warmup_steps=6,logging_steps=10,save_steps=10,save_total_limit=2,fp16=False,max_length=CONFIG['max_seq_len'],packing=False,report_to='none',seed=CONFIG['seed'])
    trainer=SFTTrainer(model=base,args=args,train_dataset=train_ds,eval_dataset=valid_ds,peft_config=lora,processing_class=tok)
    trainer.model.print_trainable_parameters(); start=time.time(); output=trainer.train(); elapsed=round(time.time()-start,2)
    trainer.model.save_pretrained(WORK/label/'adapter'); trainer.model.config.use_cache=True; trainer.model.eval()
    print('训练耗时秒：',elapsed,'步数：',trainer.state.global_step,'训练统计：',output.metrics)
    return trainer.model, trainer, elapsed

def release(*objects):
    """清理局部对象并回收显存；调用后还要删除外部持有的模型变量。"""
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 【中文语法说明】TERMS 是中文词汇表快照，RAW 是脚本生成的问答数据快照；json.loads 将文本解析为Python对象。
# BY_ID = {t["id"]: t for t in TERMS} 通过字典推导式建立“ID→词条”的查找表。
# enumerate(RAW) 同时产生下标 i 和记录；i % 2 判断奇偶，用于划分两种问法。
# 列表推导式中的多层 for 依次遍历词条和问句；标准答案 gold 只用于评分，不作为模型输入。
# 检索按名称或别名匹配；sort(..., reverse=True) 按匹配长度从大到小排序。
# 词汇表外不等于模型本来不知道；简单关键词评分也需人工复核。
TERMS = json.loads('[{"id": "ai-engineering", "term": "人工智能工程", "aliases": ["AIエンジニアリング", "AI Engineering"], "lecture": 1, "category": "整体概览", "definition": "使用已训练的基础模型作为组件，设计、构建、评价和运行实际应用；重点包括模型选择、数据与工具连接以及持续改进。", "related": ["llm", "evaluation-loop"]}, {"id": "evaluation-loop", "term": "评价与改进循环", "aliases": ["評価と改善のサイクル", "改善サイクル"], "lecture": 1, "category": "整体概览", "definition": "反复进行“构建→测量→改进”的开发流程。人工智能应用的输出会因概率性生成而变化，因此应依据评价数据和指标判断质量，再改善提示词、数据、模型和工具。本课程以学习支援智能体为例，跨课程实践这一循环。", "related": ["ai-engineering", "hallucination"]}, {"id": "llm", "term": "大语言模型（LLM）", "aliases": ["LLM（大規模言語モデル）", "大規模言語モデル", "Large Language Model"], "lecture": 1, "category": "大语言模型基础", "definition": "在大量文本上训练、通过预测输入之后的词元来生成文字的模型。改变提示词即可用于问答、摘要、代码生成等多种任务。", "related": ["token", "prompt", "pretraining"]}, {"id": "token", "term": "词元", "aliases": ["トークン", "token"], "lecture": 1, "category": "大语言模型基础", "definition": "语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。", "related": ["context-window", "rate-limit"]}, {"id": "context-window", "term": "上下文窗口", "aliases": ["コンテキストウィンドウ", "コンテキスト長", "context window"], "lecture": 1, "category": "大语言模型基础", "definition": "大语言模型一次调用中能够处理的输入与输出词元总量上限。会话历史和工具结果也需要计入，因此必须设计好传给模型的内容。", "related": ["token", "agent-loop"]}, {"id": "prompt", "term": "提示词", "aliases": ["プロンプト", "prompt"], "lecture": 1, "category": "大语言模型基础", "definition": "传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。", "related": ["system-prompt", "llm"]}, {"id": "system-prompt", "term": "系统提示词", "aliases": ["システムプロンプト", "指示文", "instructions", "system instruction"], "lecture": 1, "category": "大语言模型基础", "definition": "独立于用户问题，用于指定模型角色、语气和规则的输入；Pydantic AI可通过instructions提供。", "related": ["prompt", "guardrail"]}, {"id": "temperature", "term": "温度", "aliases": ["temperature", "温度パラメータ"], "lecture": 1, "category": "大语言模型基础", "definition": "调整选择下一个词元时随机性的设置。一般较低更稳定、较高更多样，具体支持情况取决于模型。", "related": ["llm", "evaluation-loop"]}, {"id": "hallucination", "term": "幻觉", "aliases": ["ハルシネーション", "幻覚", "hallucination"], "lecture": 1, "category": "大语言模型基础", "definition": "大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。", "related": ["tool", "evaluation-loop"]}, {"id": "api", "term": "应用程序编程接口（API）", "aliases": ["API", "Application Programming Interface", "Web API"], "lecture": 1, "category": "开发基础", "definition": "供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。", "related": ["api-key", "rate-limit"]}, {"id": "api-key", "term": "API密钥", "aliases": ["APIキー", "API key"], "lecture": 1, "category": "开发基础", "definition": "用于识别API调用者的秘密字符串。泄露后他人可能使用你的额度，所以不要直接写在代码中、提交到仓库或贴到聊天中。本课程将其登记为Codespaces secret，再通过环境变量GEMINI_API_KEY读取。", "related": ["codespaces-secret", "environment-variable"]}, {"id": "environment-variable", "term": "环境变量", "aliases": ["環境変数", "environment variable", "env"], "lecture": 1, "category": "开发基础", "definition": "在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。", "related": ["api-key", "codespaces-secret"]}, {"id": "rate-limit", "term": "速率限制", "aliases": ["レート制限", "rate limit", "429エラー"], "lecture": 1, "category": "开发基础", "definition": "单位时间内允许的请求数或词元数上限。超过时可能返回HTTP 429等错误。免费额度通常限制较严格；智能体可能为一个问题多次调用模型，因此更容易触及上限。", "related": ["api", "agent-loop", "default-model"]}, {"id": "codespaces", "term": "GitHub Codespaces", "aliases": ["GitHub Codespaces", "Codespaces", "コードスペース"], "lecture": 1, "category": "开发环境", "definition": "GitHub提供的云端开发环境，可通过浏览器使用；个人账户的免费额度有限。", "related": ["devcontainer", "codespaces-secret"]}, {"id": "devcontainer", "term": "开发容器", "aliases": ["devcontainer", "Dev Container", "devcontainer.json"], "lecture": 1, "category": "开发环境", "definition": "使用容器定义开发环境的机制，可在配置文件中指定Python版本、扩展和启动命令。", "related": ["codespaces"]}, {"id": "codespaces-secret", "term": "Codespaces秘密配置", "aliases": ["Codespaces secret", "Codespacesのシークレット"], "lecture": 1, "category": "开发环境", "definition": "在GitHub账户设置中登记的秘密值。启动指定仓库的Codespaces时会作为环境变量传入；登记时需要选择自己的派生仓库，如果在运行中登记，则需要重新启动Codespaces。", "related": ["api-key", "environment-variable", "codespaces"]}, {"id": "fork", "term": "派生副本", "aliases": ["fork", "フォーク"], "lecture": 1, "category": "Git/GitHub", "definition": "将他人的仓库完整复制到自己的账户，形成关联副本。本课程在自己的副本中完成工作和提交。", "related": ["branch", "pull-request"]}, {"id": "branch", "term": "分支", "aliases": ["ブランチ", "branch"], "lecture": 1, "category": "Git/GitHub", "definition": "用于分离工作版本的机制，使人能够不直接修改main就试验改动，完成后通过PR合入。本课程每回创建类似lec01的工作分支。", "related": ["pull-request", "fork"]}, {"id": "pull-request", "term": "拉取请求（PR）", "aliases": ["プルリクエスト（PR）", "PR", "Pull Request"], "lecture": 1, "category": "Git/GitHub", "definition": "请求把一个分支的改动合入另一个分支的机制，支持差异检查、讨论和测试。", "related": ["branch", "answer-md"]}, {"id": "github-copilot", "term": "GitHub Copilot", "aliases": ["GitHub Copilot", "Copilot"], "lecture": 1, "category": "AI编程", "definition": "GitHub的人工智能编程辅助工具，支持编辑器代码补全、聊天问答和智能体模式下的多文件编辑。Copilot Free对补全和聊天用量设有限制。", "related": ["code-completion", "coding-agent"]}, {"id": "code-completion", "term": "代码补全", "aliases": ["コード補完", "インライン補完", "inline suggestion"], "lecture": 1, "category": "AI编程", "definition": "AI根据未完成的代码或注释建议后续代码的功能，通常可用Tab键接受。建议不一定正确，需要阅读并理解后再采用。", "related": ["github-copilot"]}, {"id": "coding-agent", "term": "编程智能体", "aliases": ["コーディングエージェント", "Coding Agent", "AIコーディングエージェント"], "lecture": 1, "category": "AI编程", "definition": "使用文件读写和执行命令等工具，自主分多步完成编程任务的人工智能系统。例子包括Claude Code、Copilot智能体模式和Pi，其基本结构是语言模型、循环和工具的组合。", "related": ["ai-agent", "harness"]}, {"id": "pretraining", "term": "预训练", "aliases": ["事前学習", "pre-training", "プレトレーニング"], "lecture": 2, "category": "微调", "definition": "使用大量通用文本训练模型预测下一个词元的初始阶段。模型在此过程中获得了大量语言知识和一般常识。", "related": ["fine-tuning", "llm"]}, {"id": "fine-tuning", "term": "微调", "aliases": ["ファインチューニング", "fine-tuning", "追加学習", "FT"], "lecture": 2, "category": "微调", "definition": "使用适合特定任务或格式的数据，继续训练已经过预训练的模型。适用于调整输出格式、语气或特定领域的行为；对于经常更新的事实，使用工具或检索增强生成（RAG）参考外部资料往往更合适。", "related": ["sft", "peft", "lora"]}, {"id": "sft", "term": "监督微调（SFT）", "aliases": ["教師ありファインチューニング（SFT）", "SFT", "Supervised Fine-Tuning", "指示チューニング"], "lecture": 2, "category": "微调", "definition": "使用输入与期望输出的配对数据（例如问题与示范答案）继续训练模型的方法，是使模型学会遵循指令的代表性方法。", "related": ["fine-tuning", "lora"]}, {"id": "peft", "term": "参数高效微调（PEFT）", "aliases": ["PEFT", "Parameter-Efficient Fine-Tuning", "パラメータ効率の良いファインチューニング"], "lecture": 2, "category": "微调", "definition": "不训练模型全部参数，而只训练少量原有参数或新增的小型参数的方法总称，可以显著减少所需GPU显存和保存空间。LoRA是代表方法。", "related": ["lora", "fine-tuning"]}, {"id": "lora", "term": "低秩适应（LoRA）", "aliases": ["LoRA", "Low-Rank Adaptation"], "lecture": 2, "category": "微调", "definition": "一种参数高效微调方法：固定预训练权重，用两个小矩阵的乘积（低秩矩阵）表示权重更新量，只训练这两个小矩阵。可训练参数较少，训练结果（适配器）也可以保存为较小文件。本课程第2回演习使用LoRA。", "related": ["peft", "lora-rank", "qlora"]}, {"id": "lora-rank", "term": "低秩适应的秩", "aliases": ["ランク（LoRAのr）", "rank", "r"], "lecture": 2, "category": "微调", "definition": "决定LoRA所用低秩矩阵大小的超参数。数值越大，通常可表示的变化越丰富，但可训练参数和显存需求也会增加。", "related": ["lora"]}, {"id": "qlora", "term": "量化低秩适应（QLoRA）", "aliases": ["QLoRA", "Quantized LoRA"], "lecture": 2, "category": "微调", "definition": "将基础模型权重量化为4位等低精度以节省内存，再使用LoRA训练的方法。这样可在较小GPU上处理更大模型。原词汇表说明课程第2回因模型能放入GPU而使用LoRA，拓展部分比较4位加载的内存和输出差异。", "related": ["lora", "quantization"]}, {"id": "quantization", "term": "量化", "aliases": ["量子化", "quantization"], "lecture": 2, "category": "微调", "definition": "使用更少的位数表示模型权重，例如从16位变成4位，以减少存储等资源消耗；量化误差可能影响模型表现。", "related": ["qlora"]}, {"id": "learning-rate", "term": "学习率", "aliases": ["学習率", "learning rate", "lr"], "lecture": 2, "category": "微调", "definition": "决定每次更新时参数改变幅度的超参数。过大可能导致训练不稳定，过小则可能使训练进展缓慢。", "related": ["epoch", "overfitting"]}, {"id": "epoch", "term": "训练轮次", "aliases": ["エポック", "epoch"], "lecture": 2, "category": "微调", "definition": "训练数据整体被学习一遍称为一个训练轮次。轮数过多容易导致过拟合。", "related": ["overfitting", "learning-rate"]}, {"id": "overfitting", "term": "过拟合", "aliases": ["過学習", "overfitting", "オーバーフィッティング"], "lecture": 2, "category": "微调", "definition": "模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。", "related": ["epoch", "evaluation-loop"]}, {"id": "huggingface-hub", "term": "Hugging Face Hub", "aliases": ["Hugging Face Hub", "Hugging Face", "HF Hub"], "lecture": 2, "category": "微调", "definition": "分享和下载公开模型与数据集的平台，可用于获取开放权重模型或发布LoRA适配器。", "related": ["lora", "pretraining"]}, {"id": "ai-agent", "term": "人工智能智能体", "aliases": ["AIエージェント", "エージェント", "agent"], "lecture": 3, "category": "智能体", "definition": "模型结合工具与反馈，多轮选择行动以完成目标的系统。", "related": ["agent-loop", "tool", "harness"]}, {"id": "agent-loop", "term": "智能体循环", "aliases": ["エージェントループ", "agent loop", "ツール呼び出しループ"], "lecture": 3, "category": "智能体", "definition": "将输入交给模型；如果模型请求调用工具，程序执行工具并把结果返回模型。这个过程重复进行，直到模型返回最终回答。一次问题可能多次调用大语言模型，因此会影响时间、费用和速率限制。", "related": ["ai-agent", "tool", "rate-limit"]}, {"id": "tool", "term": "工具调用", "aliases": ["ツール（Tool use / Function calling）", "ツール呼び出し", "Function calling", "Tool use", "ツール"], "lecture": 3, "category": "智能体", "definition": "程序提供给模型调用的函数。模型根据工具名称、说明和参数类型决定何时调用以及使用什么参数，实际执行由程序负责。常用于搜索、计算、文件操作和调用外部API。", "related": ["tool-description", "agent-loop"]}, {"id": "tool-description", "term": "工具说明", "aliases": ["ツールの説明文", "docstring", "ツールスキーマ", "tool schema"], "lecture": 3, "category": "智能体", "definition": "向模型说明工具用途及参数含义的文字。在Pydantic AI中可根据函数文档字符串和类型提示自动生成；说明方式会影响模型是否调用工具及参数选择。", "related": ["tool", "pydantic-ai"]}, {"id": "structured-output", "term": "结构化输出", "aliases": ["構造化出力", "structured output", "JSON出力"], "lecture": 3, "category": "智能体", "definition": "让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。", "related": ["pydantic-ai"]}, {"id": "harness", "term": "智能体运行框架", "aliases": ["ハーネス", "agent harness", "エージェントハーネス"], "lecture": 3, "category": "智能体", "definition": "围绕模型提供循环、工具与界面的程序，使模型能够执行实际任务。", "related": ["coding-agent", "pi"]}, {"id": "pi", "term": "Pi", "aliases": ["Pi", "Pi coding agent", "pi agent"], "lecture": 3, "category": "智能体", "definition": "在终端运行的MIT许可开源编程智能体，设计为最小功能与系统提示词，由用户扩展。原资料记载2026年10月发布1.0，npm包名为@earendil-works/pi-coding-agent。", "related": ["harness", "coding-agent"]}, {"id": "pydantic-ai", "term": "Pydantic AI", "aliases": ["Pydantic AI", "pydantic-ai", "PydanticAI"], "lecture": 3, "category": "智能体", "definition": "使用Python构建智能体的框架，支持类型化工具与输出，并可用测试模型验证流程。", "related": ["test-model", "structured-output", "adk"]}, {"id": "adk", "term": "智能体开发工具包（ADK）", "aliases": ["ADK（Agent Development Kit）", "ADK", "Agent Development Kit", "google-adk"], "lecture": 3, "category": "智能体", "definition": "Google公开的开源智能体开发框架，与Gemini集成较深，可通过adk web查看工具调用流程。原资料指出2.0相对1.x包含破坏性变化；本课程在拓展作业中涉及。", "related": ["pydantic-ai", "ai-agent"]}, {"id": "mcp", "term": "模型上下文协议（MCP）", "aliases": ["MCP（Model Context Protocol）", "MCP", "Model Context Protocol"], "lecture": 3, "category": "智能体", "definition": "连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。", "related": ["tool", "ai-agent", "mcp-server", "mcp-client"]}, {"id": "guardrail", "term": "防护措施", "aliases": ["ガードレール", "guardrail"], "lecture": 3, "category": "智能体", "definition": "为防止智能体产生不合适行为或输出而设置的限制和检查，包括指令规则、工具权限限制、输出检查和重要操作前的人类确认。", "related": ["human-in-the-loop", "system-prompt"]}, {"id": "human-in-the-loop", "term": "人在回路", "aliases": ["Human-in-the-loop", "HITL", "人間による承認", "確認を挟む"], "lecture": 3, "category": "智能体", "definition": "在智能体处理过程中加入人工检查或批准，常用于写入数据或发送信息等不易撤销的操作。Pydantic AI可为工具设置requires_approval=True。", "related": ["guardrail", "tool"]}, {"id": "test-model", "term": "测试模型", "aliases": ["TestModel", "テスト用モデル"], "lecture": 3, "category": "智能体", "definition": "用于测试的模拟模型，不调用真实语言模型即可检查工具连接和流程。", "related": ["pydantic-ai", "rate-limit"]}, {"id": "homework-levels", "term": "作业的三个级别", "aliases": ["宿題の3レベル", "3レベル", "宿題レベル"], "lecture": 1, "category": "课程", "definition": "AIE2026作业分为三个级别：基础（确认讲义内容）、实现（自己编写代码）、拓展（调查或尝试）。每回完成其中一个即可；原词汇表说明原则上不评分。", "related": ["answer-md"]}, {"id": "answer-md", "term": "answer.md", "aliases": ["answer.md", "解答ファイル"], "lecture": 1, "category": "课程", "definition": "AIE2026各回记录演习与作业结果的文件，位于lectures/NN/answer.md。填写选择的级别、做法、结果和发现，再从工作分支向自己派生仓库的main分支创建PR提交。", "related": ["pull-request", "homework-levels"]}, {"id": "lec-start-tag", "term": "lecNN-start标签", "aliases": ["lecNN-start タグ", "開始タグ", "lec03-start"], "lecture": 1, "category": "课程", "definition": "课程仓库中标记每次演习开始状态的Git标签，例如lec03-start。即使前一回尚未完成，也可以从相应标签创建分支并开始该回演习。", "related": ["branch"]}, {"id": "default-model", "term": "默认模型", "aliases": ["既定モデル", "デフォルトモデル", "gemini-3.5-flash-lite"], "lecture": 1, "category": "课程", "definition": "原词汇表所记载的课程演习默认模型为gemini-3.5-flash-lite，考虑了免费额度的速率限制和课程运行稳定性。可以通过环境变量AIE_MODEL切换模型，也可以使用自己签约的API；实际可用性应以当前服务为准。", "related": ["rate-limit", "api-key"]}, {"id": "external-tool", "term": "外部工具", "aliases": ["外部ツール", "external tool", "外部サービス"], "lecture": 5, "category": "MCP", "definition": "程序进程之外的功能，例如网页获取、数据库或外部API，可作为智能体工具调用。", "related": ["tool", "mcp", "mcp-server"]}, {"id": "mcp-server", "term": "MCP服务器", "aliases": ["MCPサーバー", "MCP server"], "lecture": 5, "category": "MCP", "definition": "按照MCP协议公开工具、资源或提示词的程序。服务器公开工具名称、说明和参数结构，接收客户端请求，执行工具并返回结果。本课程示例在app/mcp_server.py中公开词汇表搜索。", "related": ["mcp", "mcp-client", "mcp-transport", "fetch-server"]}, {"id": "mcp-client", "term": "MCP客户端", "aliases": ["MCPクライアント", "MCP client", "MCPホスト"], "lecture": 5, "category": "MCP", "definition": "连接MCP服务器、将公开工具提供给模型，并根据模型请求调用工具的一方。智能体程序、编辑器或应用都可以作为客户端；多个兼容客户端可复用同一服务器。", "related": ["mcp", "mcp-server", "mcp-config"]}, {"id": "mcp-transport", "term": "传输方式", "aliases": ["トランスポート（stdio・Streamable HTTP）", "stdio", "Streamable HTTP", "transport", "MCPのトランスポート"], "lecture": 5, "category": "MCP", "definition": "MCP客户端与服务器交换消息的通信路径。stdio方式由客户端启动子进程，通过标准输入输出通信；Streamable HTTP允许通过URL跨进程或跨机器连接，需考虑公开范围和认证。", "related": ["mcp-server", "mcp-client"]}, {"id": "mcp-config", "term": "mcpServers配置", "aliases": ["mcpServers設定", "mcpServers", "MCPの設定ファイル", "mcp_servers.json", "mcp.json"], "lecture": 5, "category": "MCP", "definition": "列出所用MCP服务器的配置文件，为每个服务器记录名称，以及stdio启动命令或HTTP连接地址。原词汇表列出Claude Desktop、Cursor和Pydantic AI等使用mcpServers格式；VS Code / Copilot Chat使用servers格式。本课程示例文件为app/mcp_servers.json。", "related": ["mcp-client", "tool-namespace"]}, {"id": "tool-namespace", "term": "工具命名空间", "aliases": ["ツールの名前空間", "ツール名のプレフィックス", "prefix", "名前の衝突"], "lecture": 5, "category": "MCP", "definition": "给不同服务器的工具添加前缀，以区分重名工具。", "related": ["mcp-config", "tool-description", "rate-limit"]}, {"id": "fetch-server", "term": "fetch服务器", "aliases": ["fetchサーバー", "mcp-server-fetch", "fetch", "Web取得ツール"], "lecture": 5, "category": "MCP", "definition": "获取指定URL页面、将正文转换为Markdown返回的现成MCP服务器，可通过uvx mcp-server-fetch启动。它展示了通过配置增加外部工具的方式；使用者应了解模型访问的网址及发送的内容。", "related": ["mcp-server", "external-tool", "guardrail"]}]')
RAW = json.loads('[{"messages": [{"role": "user", "content": "“人工智能工程”是什么意思？"}, {"role": "assistant", "content": "使用已训练的基础模型作为组件，设计、构建、评价和运行实际应用；重点包括模型选择、数据与工具连接以及持续改进。"}]}, {"messages": [{"role": "user", "content": "请简要解释人工智能工程。"}, {"role": "assistant", "content": "使用已训练的基础模型作为组件，设计、构建、评价和运行实际应用；重点包括模型选择、数据与工具连接以及持续改进。"}]}, {"messages": [{"role": "user", "content": "“评价与改进循环”是什么意思？"}, {"role": "assistant", "content": "反复进行“构建→测量→改进”的开发流程。人工智能应用的输出会因概率性生成而变化，因此应依据评价数据和指标判断质量，再改善提示词、数据、模型和工具。本课程以学习支援智能体为例，跨课程实践这一循环。"}]}, {"messages": [{"role": "user", "content": "请简要解释评价与改进循环。"}, {"role": "assistant", "content": "反复进行“构建→测量→改进”的开发流程。人工智能应用的输出会因概率性生成而变化，因此应依据评价数据和指标判断质量，再改善提示词、数据、模型和工具。本课程以学习支援智能体为例，跨课程实践这一循环。"}]}, {"messages": [{"role": "user", "content": "“大语言模型（LLM）”是什么意思？"}, {"role": "assistant", "content": "在大量文本上训练、通过预测输入之后的词元来生成文字的模型。改变提示词即可用于问答、摘要、代码生成等多种任务。"}]}, {"messages": [{"role": "user", "content": "请简要解释大语言模型（LLM）。"}, {"role": "assistant", "content": "在大量文本上训练、通过预测输入之后的词元来生成文字的模型。改变提示词即可用于问答、摘要、代码生成等多种任务。"}]}, {"messages": [{"role": "user", "content": "“词元”是什么意思？"}, {"role": "assistant", "content": "语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。"}]}, {"messages": [{"role": "user", "content": "请简要解释词元。"}, {"role": "assistant", "content": "语言模型处理文本的单位，可以是单词或字符片段；费用、限流和上下文长度通常按词元计数。"}]}, {"messages": [{"role": "user", "content": "“上下文窗口”是什么意思？"}, {"role": "assistant", "content": "大语言模型一次调用中能够处理的输入与输出词元总量上限。会话历史和工具结果也需要计入，因此必须设计好传给模型的内容。"}]}, {"messages": [{"role": "user", "content": "请简要解释上下文窗口。"}, {"role": "assistant", "content": "大语言模型一次调用中能够处理的输入与输出词元总量上限。会话历史和工具结果也需要计入，因此必须设计好传给模型的内容。"}]}, {"messages": [{"role": "user", "content": "“提示词”是什么意思？"}, {"role": "assistant", "content": "传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。"}]}, {"messages": [{"role": "user", "content": "请简要解释提示词。"}, {"role": "assistant", "content": "传给大语言模型的输入文字。除了问题，还可以包含角色、限制、输出格式和示例，以引导模型输出。"}]}, {"messages": [{"role": "user", "content": "“系统提示词”是什么意思？"}, {"role": "assistant", "content": "独立于用户问题，用于指定模型角色、语气和规则的输入；Pydantic AI可通过instructions提供。"}]}, {"messages": [{"role": "user", "content": "请简要解释系统提示词。"}, {"role": "assistant", "content": "独立于用户问题，用于指定模型角色、语气和规则的输入；Pydantic AI可通过instructions提供。"}]}, {"messages": [{"role": "user", "content": "“温度”是什么意思？"}, {"role": "assistant", "content": "调整选择下一个词元时随机性的设置。一般较低更稳定、较高更多样，具体支持情况取决于模型。"}]}, {"messages": [{"role": "user", "content": "请简要解释温度。"}, {"role": "assistant", "content": "调整选择下一个词元时随机性的设置。一般较低更稳定、较高更多样，具体支持情况取决于模型。"}]}, {"messages": [{"role": "user", "content": "“幻觉”是什么意思？"}, {"role": "assistant", "content": "大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。"}]}, {"messages": [{"role": "user", "content": "请简要解释幻觉。"}, {"role": "assistant", "content": "大语言模型将不符合事实或不存在的信息生成得似乎可信的现象。在询问模型不知道的最新信息、组织或课程专属信息时可能出现；可通过工具访问外部资料、要求提供依据和评价检测等方法降低风险。"}]}, {"messages": [{"role": "user", "content": "“应用程序编程接口（API）”是什么意思？"}, {"role": "assistant", "content": "供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。"}]}, {"messages": [{"role": "user", "content": "请简要解释应用程序编程接口（API）。"}, {"role": "assistant", "content": "供程序调用其他服务功能的接口。本课程通过Python调用Gemini API以使用大语言模型。"}]}, {"messages": [{"role": "user", "content": "“API密钥”是什么意思？"}, {"role": "assistant", "content": "用于识别API调用者的秘密字符串。泄露后他人可能使用你的额度，所以不要直接写在代码中、提交到仓库或贴到聊天中。本课程将其登记为Codespaces secret，再通过环境变量GEMINI_API_KEY读取。"}]}, {"messages": [{"role": "user", "content": "请简要解释API密钥。"}, {"role": "assistant", "content": "用于识别API调用者的秘密字符串。泄露后他人可能使用你的额度，所以不要直接写在代码中、提交到仓库或贴到聊天中。本课程将其登记为Codespaces secret，再通过环境变量GEMINI_API_KEY读取。"}]}, {"messages": [{"role": "user", "content": "“环境变量”是什么意思？"}, {"role": "assistant", "content": "在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。"}]}, {"messages": [{"role": "user", "content": "请简要解释环境变量。"}, {"role": "assistant", "content": "在程序外部（操作系统或运行环境）设置、由程序通过名称读取的值。它可将API密钥等秘密信息和不同环境下变化的配置与代码分开。"}]}, {"messages": [{"role": "user", "content": "“速率限制”是什么意思？"}, {"role": "assistant", "content": "单位时间内允许的请求数或词元数上限。超过时可能返回HTTP 429等错误。免费额度通常限制较严格；智能体可能为一个问题多次调用模型，因此更容易触及上限。"}]}, {"messages": [{"role": "user", "content": "请简要解释速率限制。"}, {"role": "assistant", "content": "单位时间内允许的请求数或词元数上限。超过时可能返回HTTP 429等错误。免费额度通常限制较严格；智能体可能为一个问题多次调用模型，因此更容易触及上限。"}]}, {"messages": [{"role": "user", "content": "“GitHub Codespaces”是什么意思？"}, {"role": "assistant", "content": "GitHub提供的云端开发环境，可通过浏览器使用；个人账户的免费额度有限。"}]}, {"messages": [{"role": "user", "content": "请简要解释GitHub Codespaces。"}, {"role": "assistant", "content": "GitHub提供的云端开发环境，可通过浏览器使用；个人账户的免费额度有限。"}]}, {"messages": [{"role": "user", "content": "“开发容器”是什么意思？"}, {"role": "assistant", "content": "使用容器定义开发环境的机制，可在配置文件中指定Python版本、扩展和启动命令。"}]}, {"messages": [{"role": "user", "content": "请简要解释开发容器。"}, {"role": "assistant", "content": "使用容器定义开发环境的机制，可在配置文件中指定Python版本、扩展和启动命令。"}]}, {"messages": [{"role": "user", "content": "“Codespaces秘密配置”是什么意思？"}, {"role": "assistant", "content": "在GitHub账户设置中登记的秘密值。启动指定仓库的Codespaces时会作为环境变量传入；登记时需要选择自己的派生仓库，如果在运行中登记，则需要重新启动Codespaces。"}]}, {"messages": [{"role": "user", "content": "请简要解释Codespaces秘密配置。"}, {"role": "assistant", "content": "在GitHub账户设置中登记的秘密值。启动指定仓库的Codespaces时会作为环境变量传入；登记时需要选择自己的派生仓库，如果在运行中登记，则需要重新启动Codespaces。"}]}, {"messages": [{"role": "user", "content": "“派生副本”是什么意思？"}, {"role": "assistant", "content": "将他人的仓库完整复制到自己的账户，形成关联副本。本课程在自己的副本中完成工作和提交。"}]}, {"messages": [{"role": "user", "content": "请简要解释派生副本。"}, {"role": "assistant", "content": "将他人的仓库完整复制到自己的账户，形成关联副本。本课程在自己的副本中完成工作和提交。"}]}, {"messages": [{"role": "user", "content": "“分支”是什么意思？"}, {"role": "assistant", "content": "用于分离工作版本的机制，使人能够不直接修改main就试验改动，完成后通过PR合入。本课程每回创建类似lec01的工作分支。"}]}, {"messages": [{"role": "user", "content": "请简要解释分支。"}, {"role": "assistant", "content": "用于分离工作版本的机制，使人能够不直接修改main就试验改动，完成后通过PR合入。本课程每回创建类似lec01的工作分支。"}]}, {"messages": [{"role": "user", "content": "“拉取请求（PR）”是什么意思？"}, {"role": "assistant", "content": "请求把一个分支的改动合入另一个分支的机制，支持差异检查、讨论和测试。"}]}, {"messages": [{"role": "user", "content": "请简要解释拉取请求（PR）。"}, {"role": "assistant", "content": "请求把一个分支的改动合入另一个分支的机制，支持差异检查、讨论和测试。"}]}, {"messages": [{"role": "user", "content": "“GitHub Copilot”是什么意思？"}, {"role": "assistant", "content": "GitHub的人工智能编程辅助工具，支持编辑器代码补全、聊天问答和智能体模式下的多文件编辑。Copilot Free对补全和聊天用量设有限制。"}]}, {"messages": [{"role": "user", "content": "请简要解释GitHub Copilot。"}, {"role": "assistant", "content": "GitHub的人工智能编程辅助工具，支持编辑器代码补全、聊天问答和智能体模式下的多文件编辑。Copilot Free对补全和聊天用量设有限制。"}]}, {"messages": [{"role": "user", "content": "“代码补全”是什么意思？"}, {"role": "assistant", "content": "AI根据未完成的代码或注释建议后续代码的功能，通常可用Tab键接受。建议不一定正确，需要阅读并理解后再采用。"}]}, {"messages": [{"role": "user", "content": "请简要解释代码补全。"}, {"role": "assistant", "content": "AI根据未完成的代码或注释建议后续代码的功能，通常可用Tab键接受。建议不一定正确，需要阅读并理解后再采用。"}]}, {"messages": [{"role": "user", "content": "“编程智能体”是什么意思？"}, {"role": "assistant", "content": "使用文件读写和执行命令等工具，自主分多步完成编程任务的人工智能系统。例子包括Claude Code、Copilot智能体模式和Pi，其基本结构是语言模型、循环和工具的组合。"}]}, {"messages": [{"role": "user", "content": "请简要解释编程智能体。"}, {"role": "assistant", "content": "使用文件读写和执行命令等工具，自主分多步完成编程任务的人工智能系统。例子包括Claude Code、Copilot智能体模式和Pi，其基本结构是语言模型、循环和工具的组合。"}]}, {"messages": [{"role": "user", "content": "“预训练”是什么意思？"}, {"role": "assistant", "content": "使用大量通用文本训练模型预测下一个词元的初始阶段。模型在此过程中获得了大量语言知识和一般常识。"}]}, {"messages": [{"role": "user", "content": "请简要解释预训练。"}, {"role": "assistant", "content": "使用大量通用文本训练模型预测下一个词元的初始阶段。模型在此过程中获得了大量语言知识和一般常识。"}]}, {"messages": [{"role": "user", "content": "“微调”是什么意思？"}, {"role": "assistant", "content": "使用适合特定任务或格式的数据，继续训练已经过预训练的模型。适用于调整输出格式、语气或特定领域的行为；对于经常更新的事实，使用工具或检索增强生成（RAG）参考外部资料往往更合适。"}]}, {"messages": [{"role": "user", "content": "请简要解释微调。"}, {"role": "assistant", "content": "使用适合特定任务或格式的数据，继续训练已经过预训练的模型。适用于调整输出格式、语气或特定领域的行为；对于经常更新的事实，使用工具或检索增强生成（RAG）参考外部资料往往更合适。"}]}, {"messages": [{"role": "user", "content": "“监督微调（SFT）”是什么意思？"}, {"role": "assistant", "content": "使用输入与期望输出的配对数据（例如问题与示范答案）继续训练模型的方法，是使模型学会遵循指令的代表性方法。"}]}, {"messages": [{"role": "user", "content": "请简要解释监督微调（SFT）。"}, {"role": "assistant", "content": "使用输入与期望输出的配对数据（例如问题与示范答案）继续训练模型的方法，是使模型学会遵循指令的代表性方法。"}]}, {"messages": [{"role": "user", "content": "“参数高效微调（PEFT）”是什么意思？"}, {"role": "assistant", "content": "不训练模型全部参数，而只训练少量原有参数或新增的小型参数的方法总称，可以显著减少所需GPU显存和保存空间。LoRA是代表方法。"}]}, {"messages": [{"role": "user", "content": "请简要解释参数高效微调（PEFT）。"}, {"role": "assistant", "content": "不训练模型全部参数，而只训练少量原有参数或新增的小型参数的方法总称，可以显著减少所需GPU显存和保存空间。LoRA是代表方法。"}]}, {"messages": [{"role": "user", "content": "“低秩适应（LoRA）”是什么意思？"}, {"role": "assistant", "content": "一种参数高效微调方法：固定预训练权重，用两个小矩阵的乘积（低秩矩阵）表示权重更新量，只训练这两个小矩阵。可训练参数较少，训练结果（适配器）也可以保存为较小文件。本课程第2回演习使用LoRA。"}]}, {"messages": [{"role": "user", "content": "请简要解释低秩适应（LoRA）。"}, {"role": "assistant", "content": "一种参数高效微调方法：固定预训练权重，用两个小矩阵的乘积（低秩矩阵）表示权重更新量，只训练这两个小矩阵。可训练参数较少，训练结果（适配器）也可以保存为较小文件。本课程第2回演习使用LoRA。"}]}, {"messages": [{"role": "user", "content": "“低秩适应的秩”是什么意思？"}, {"role": "assistant", "content": "决定LoRA所用低秩矩阵大小的超参数。数值越大，通常可表示的变化越丰富，但可训练参数和显存需求也会增加。"}]}, {"messages": [{"role": "user", "content": "请简要解释低秩适应的秩。"}, {"role": "assistant", "content": "决定LoRA所用低秩矩阵大小的超参数。数值越大，通常可表示的变化越丰富，但可训练参数和显存需求也会增加。"}]}, {"messages": [{"role": "user", "content": "“量化低秩适应（QLoRA）”是什么意思？"}, {"role": "assistant", "content": "将基础模型权重量化为4位等低精度以节省内存，再使用LoRA训练的方法。这样可在较小GPU上处理更大模型。原词汇表说明课程第2回因模型能放入GPU而使用LoRA，拓展部分比较4位加载的内存和输出差异。"}]}, {"messages": [{"role": "user", "content": "请简要解释量化低秩适应（QLoRA）。"}, {"role": "assistant", "content": "将基础模型权重量化为4位等低精度以节省内存，再使用LoRA训练的方法。这样可在较小GPU上处理更大模型。原词汇表说明课程第2回因模型能放入GPU而使用LoRA，拓展部分比较4位加载的内存和输出差异。"}]}, {"messages": [{"role": "user", "content": "“量化”是什么意思？"}, {"role": "assistant", "content": "使用更少的位数表示模型权重，例如从16位变成4位，以减少存储等资源消耗；量化误差可能影响模型表现。"}]}, {"messages": [{"role": "user", "content": "请简要解释量化。"}, {"role": "assistant", "content": "使用更少的位数表示模型权重，例如从16位变成4位，以减少存储等资源消耗；量化误差可能影响模型表现。"}]}, {"messages": [{"role": "user", "content": "“学习率”是什么意思？"}, {"role": "assistant", "content": "决定每次更新时参数改变幅度的超参数。过大可能导致训练不稳定，过小则可能使训练进展缓慢。"}]}, {"messages": [{"role": "user", "content": "请简要解释学习率。"}, {"role": "assistant", "content": "决定每次更新时参数改变幅度的超参数。过大可能导致训练不稳定，过小则可能使训练进展缓慢。"}]}, {"messages": [{"role": "user", "content": "“训练轮次”是什么意思？"}, {"role": "assistant", "content": "训练数据整体被学习一遍称为一个训练轮次。轮数过多容易导致过拟合。"}]}, {"messages": [{"role": "user", "content": "请简要解释训练轮次。"}, {"role": "assistant", "content": "训练数据整体被学习一遍称为一个训练轮次。轮数过多容易导致过拟合。"}]}, {"messages": [{"role": "user", "content": "“过拟合”是什么意思？"}, {"role": "assistant", "content": "模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。"}]}, {"messages": [{"role": "user", "content": "请简要解释过拟合。"}, {"role": "assistant", "content": "模型过度适应训练数据、对未见输入的表现下降的状态。可以通过未用于训练的验证数据测量表现并发现这一问题。"}]}, {"messages": [{"role": "user", "content": "“Hugging Face Hub”是什么意思？"}, {"role": "assistant", "content": "分享和下载公开模型与数据集的平台，可用于获取开放权重模型或发布LoRA适配器。"}]}, {"messages": [{"role": "user", "content": "请简要解释Hugging Face Hub。"}, {"role": "assistant", "content": "分享和下载公开模型与数据集的平台，可用于获取开放权重模型或发布LoRA适配器。"}]}, {"messages": [{"role": "user", "content": "“人工智能智能体”是什么意思？"}, {"role": "assistant", "content": "模型结合工具与反馈，多轮选择行动以完成目标的系统。"}]}, {"messages": [{"role": "user", "content": "请简要解释人工智能智能体。"}, {"role": "assistant", "content": "模型结合工具与反馈，多轮选择行动以完成目标的系统。"}]}, {"messages": [{"role": "user", "content": "“智能体循环”是什么意思？"}, {"role": "assistant", "content": "将输入交给模型；如果模型请求调用工具，程序执行工具并把结果返回模型。这个过程重复进行，直到模型返回最终回答。一次问题可能多次调用大语言模型，因此会影响时间、费用和速率限制。"}]}, {"messages": [{"role": "user", "content": "请简要解释智能体循环。"}, {"role": "assistant", "content": "将输入交给模型；如果模型请求调用工具，程序执行工具并把结果返回模型。这个过程重复进行，直到模型返回最终回答。一次问题可能多次调用大语言模型，因此会影响时间、费用和速率限制。"}]}, {"messages": [{"role": "user", "content": "“工具调用”是什么意思？"}, {"role": "assistant", "content": "程序提供给模型调用的函数。模型根据工具名称、说明和参数类型决定何时调用以及使用什么参数，实际执行由程序负责。常用于搜索、计算、文件操作和调用外部API。"}]}, {"messages": [{"role": "user", "content": "请简要解释工具调用。"}, {"role": "assistant", "content": "程序提供给模型调用的函数。模型根据工具名称、说明和参数类型决定何时调用以及使用什么参数，实际执行由程序负责。常用于搜索、计算、文件操作和调用外部API。"}]}, {"messages": [{"role": "user", "content": "“工具说明”是什么意思？"}, {"role": "assistant", "content": "向模型说明工具用途及参数含义的文字。在Pydantic AI中可根据函数文档字符串和类型提示自动生成；说明方式会影响模型是否调用工具及参数选择。"}]}, {"messages": [{"role": "user", "content": "请简要解释工具说明。"}, {"role": "assistant", "content": "向模型说明工具用途及参数含义的文字。在Pydantic AI中可根据函数文档字符串和类型提示自动生成；说明方式会影响模型是否调用工具及参数选择。"}]}, {"messages": [{"role": "user", "content": "“结构化输出”是什么意思？"}, {"role": "assistant", "content": "让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。"}]}, {"messages": [{"role": "user", "content": "请简要解释结构化输出。"}, {"role": "assistant", "content": "让模型输出遵循指定类型或JSON模式的机制，便于程序后续处理。Pydantic AI中可通过output_type指定Pydantic模型。"}]}, {"messages": [{"role": "user", "content": "“智能体运行框架”是什么意思？"}, {"role": "assistant", "content": "围绕模型提供循环、工具与界面的程序，使模型能够执行实际任务。"}]}, {"messages": [{"role": "user", "content": "请简要解释智能体运行框架。"}, {"role": "assistant", "content": "围绕模型提供循环、工具与界面的程序，使模型能够执行实际任务。"}]}, {"messages": [{"role": "user", "content": "“Pi”是什么意思？"}, {"role": "assistant", "content": "在终端运行的MIT许可开源编程智能体，设计为最小功能与系统提示词，由用户扩展。原资料记载2026年10月发布1.0，npm包名为@earendil-works/pi-coding-agent。"}]}, {"messages": [{"role": "user", "content": "请简要解释Pi。"}, {"role": "assistant", "content": "在终端运行的MIT许可开源编程智能体，设计为最小功能与系统提示词，由用户扩展。原资料记载2026年10月发布1.0，npm包名为@earendil-works/pi-coding-agent。"}]}, {"messages": [{"role": "user", "content": "“Pydantic AI”是什么意思？"}, {"role": "assistant", "content": "使用Python构建智能体的框架，支持类型化工具与输出，并可用测试模型验证流程。"}]}, {"messages": [{"role": "user", "content": "请简要解释Pydantic AI。"}, {"role": "assistant", "content": "使用Python构建智能体的框架，支持类型化工具与输出，并可用测试模型验证流程。"}]}, {"messages": [{"role": "user", "content": "“智能体开发工具包（ADK）”是什么意思？"}, {"role": "assistant", "content": "Google公开的开源智能体开发框架，与Gemini集成较深，可通过adk web查看工具调用流程。原资料指出2.0相对1.x包含破坏性变化；本课程在拓展作业中涉及。"}]}, {"messages": [{"role": "user", "content": "请简要解释智能体开发工具包（ADK）。"}, {"role": "assistant", "content": "Google公开的开源智能体开发框架，与Gemini集成较深，可通过adk web查看工具调用流程。原资料指出2.0相对1.x包含破坏性变化；本课程在拓展作业中涉及。"}]}, {"messages": [{"role": "user", "content": "“模型上下文协议（MCP）”是什么意思？"}, {"role": "assistant", "content": "连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。"}]}, {"messages": [{"role": "user", "content": "请简要解释模型上下文协议（MCP）。"}, {"role": "assistant", "content": "连接人工智能应用和外部工具或数据源的开放标准协议。通过MCP服务器公开的工具，可由支持协议的智能体或编辑器等客户端以共同方式使用。"}]}, {"messages": [{"role": "user", "content": "“防护措施”是什么意思？"}, {"role": "assistant", "content": "为防止智能体产生不合适行为或输出而设置的限制和检查，包括指令规则、工具权限限制、输出检查和重要操作前的人类确认。"}]}, {"messages": [{"role": "user", "content": "请简要解释防护措施。"}, {"role": "assistant", "content": "为防止智能体产生不合适行为或输出而设置的限制和检查，包括指令规则、工具权限限制、输出检查和重要操作前的人类确认。"}]}, {"messages": [{"role": "user", "content": "“人在回路”是什么意思？"}, {"role": "assistant", "content": "在智能体处理过程中加入人工检查或批准，常用于写入数据或发送信息等不易撤销的操作。Pydantic AI可为工具设置requires_approval=True。"}]}, {"messages": [{"role": "user", "content": "请简要解释人在回路。"}, {"role": "assistant", "content": "在智能体处理过程中加入人工检查或批准，常用于写入数据或发送信息等不易撤销的操作。Pydantic AI可为工具设置requires_approval=True。"}]}, {"messages": [{"role": "user", "content": "“测试模型”是什么意思？"}, {"role": "assistant", "content": "用于测试的模拟模型，不调用真实语言模型即可检查工具连接和流程。"}]}, {"messages": [{"role": "user", "content": "请简要解释测试模型。"}, {"role": "assistant", "content": "用于测试的模拟模型，不调用真实语言模型即可检查工具连接和流程。"}]}, {"messages": [{"role": "user", "content": "“作业的三个级别”是什么意思？"}, {"role": "assistant", "content": "AIE2026作业分为三个级别：基础（确认讲义内容）、实现（自己编写代码）、拓展（调查或尝试）。每回完成其中一个即可；原词汇表说明原则上不评分。"}]}, {"messages": [{"role": "user", "content": "请简要解释作业的三个级别。"}, {"role": "assistant", "content": "AIE2026作业分为三个级别：基础（确认讲义内容）、实现（自己编写代码）、拓展（调查或尝试）。每回完成其中一个即可；原词汇表说明原则上不评分。"}]}, {"messages": [{"role": "user", "content": "“answer.md”是什么意思？"}, {"role": "assistant", "content": "AIE2026各回记录演习与作业结果的文件，位于lectures/NN/answer.md。填写选择的级别、做法、结果和发现，再从工作分支向自己派生仓库的main分支创建PR提交。"}]}, {"messages": [{"role": "user", "content": "请简要解释answer.md。"}, {"role": "assistant", "content": "AIE2026各回记录演习与作业结果的文件，位于lectures/NN/answer.md。填写选择的级别、做法、结果和发现，再从工作分支向自己派生仓库的main分支创建PR提交。"}]}, {"messages": [{"role": "user", "content": "“lecNN-start标签”是什么意思？"}, {"role": "assistant", "content": "课程仓库中标记每次演习开始状态的Git标签，例如lec03-start。即使前一回尚未完成，也可以从相应标签创建分支并开始该回演习。"}]}, {"messages": [{"role": "user", "content": "请简要解释lecNN-start标签。"}, {"role": "assistant", "content": "课程仓库中标记每次演习开始状态的Git标签，例如lec03-start。即使前一回尚未完成，也可以从相应标签创建分支并开始该回演习。"}]}, {"messages": [{"role": "user", "content": "“默认模型”是什么意思？"}, {"role": "assistant", "content": "原词汇表所记载的课程演习默认模型为gemini-3.5-flash-lite，考虑了免费额度的速率限制和课程运行稳定性。可以通过环境变量AIE_MODEL切换模型，也可以使用自己签约的API；实际可用性应以当前服务为准。"}]}, {"messages": [{"role": "user", "content": "请简要解释默认模型。"}, {"role": "assistant", "content": "原词汇表所记载的课程演习默认模型为gemini-3.5-flash-lite，考虑了免费额度的速率限制和课程运行稳定性。可以通过环境变量AIE_MODEL切换模型，也可以使用自己签约的API；实际可用性应以当前服务为准。"}]}, {"messages": [{"role": "user", "content": "“外部工具”是什么意思？"}, {"role": "assistant", "content": "程序进程之外的功能，例如网页获取、数据库或外部API，可作为智能体工具调用。"}]}, {"messages": [{"role": "user", "content": "请简要解释外部工具。"}, {"role": "assistant", "content": "程序进程之外的功能，例如网页获取、数据库或外部API，可作为智能体工具调用。"}]}, {"messages": [{"role": "user", "content": "“MCP服务器”是什么意思？"}, {"role": "assistant", "content": "按照MCP协议公开工具、资源或提示词的程序。服务器公开工具名称、说明和参数结构，接收客户端请求，执行工具并返回结果。本课程示例在app/mcp_server.py中公开词汇表搜索。"}]}, {"messages": [{"role": "user", "content": "请简要解释MCP服务器。"}, {"role": "assistant", "content": "按照MCP协议公开工具、资源或提示词的程序。服务器公开工具名称、说明和参数结构，接收客户端请求，执行工具并返回结果。本课程示例在app/mcp_server.py中公开词汇表搜索。"}]}, {"messages": [{"role": "user", "content": "“MCP客户端”是什么意思？"}, {"role": "assistant", "content": "连接MCP服务器、将公开工具提供给模型，并根据模型请求调用工具的一方。智能体程序、编辑器或应用都可以作为客户端；多个兼容客户端可复用同一服务器。"}]}, {"messages": [{"role": "user", "content": "请简要解释MCP客户端。"}, {"role": "assistant", "content": "连接MCP服务器、将公开工具提供给模型，并根据模型请求调用工具的一方。智能体程序、编辑器或应用都可以作为客户端；多个兼容客户端可复用同一服务器。"}]}, {"messages": [{"role": "user", "content": "“传输方式”是什么意思？"}, {"role": "assistant", "content": "MCP客户端与服务器交换消息的通信路径。stdio方式由客户端启动子进程，通过标准输入输出通信；Streamable HTTP允许通过URL跨进程或跨机器连接，需考虑公开范围和认证。"}]}, {"messages": [{"role": "user", "content": "请简要解释传输方式。"}, {"role": "assistant", "content": "MCP客户端与服务器交换消息的通信路径。stdio方式由客户端启动子进程，通过标准输入输出通信；Streamable HTTP允许通过URL跨进程或跨机器连接，需考虑公开范围和认证。"}]}, {"messages": [{"role": "user", "content": "“mcpServers配置”是什么意思？"}, {"role": "assistant", "content": "列出所用MCP服务器的配置文件，为每个服务器记录名称，以及stdio启动命令或HTTP连接地址。原词汇表列出Claude Desktop、Cursor和Pydantic AI等使用mcpServers格式；VS Code / Copilot Chat使用servers格式。本课程示例文件为app/mcp_servers.json。"}]}, {"messages": [{"role": "user", "content": "请简要解释mcpServers配置。"}, {"role": "assistant", "content": "列出所用MCP服务器的配置文件，为每个服务器记录名称，以及stdio启动命令或HTTP连接地址。原词汇表列出Claude Desktop、Cursor和Pydantic AI等使用mcpServers格式；VS Code / Copilot Chat使用servers格式。本课程示例文件为app/mcp_servers.json。"}]}, {"messages": [{"role": "user", "content": "“工具命名空间”是什么意思？"}, {"role": "assistant", "content": "给不同服务器的工具添加前缀，以区分重名工具。"}]}, {"messages": [{"role": "user", "content": "请简要解释工具命名空间。"}, {"role": "assistant", "content": "给不同服务器的工具添加前缀，以区分重名工具。"}]}, {"messages": [{"role": "user", "content": "“fetch服务器”是什么意思？"}, {"role": "assistant", "content": "获取指定URL页面、将正文转换为Markdown返回的现成MCP服务器，可通过uvx mcp-server-fetch启动。它展示了通过配置增加外部工具的方式；使用者应了解模型访问的网址及发送的内容。"}]}, {"messages": [{"role": "user", "content": "请简要解释fetch服务器。"}, {"role": "assistant", "content": "获取指定URL页面、将正文转换为Markdown返回的现成MCP服务器，可通过uvx mcp-server-fetch启动。它展示了通过配置增加外部工具的方式；使用者应了解模型访问的网址及发送的内容。"}]}]')
def example(question, answer, review, group, source, context, keywords):
    """构造问题和标准答案；评价前会剥离答案，只向模型传问题。"""
    return {"question":question,"answer":answer,"needs_review":review,"group":group,"source":source,"context":context,"keywords":keywords}

# 数据来源：默认使用笔记本内嵌快照；如需读取Colab上传的最新文件，改成“上传”。
DATA_MODE = "上传"  # 可选值："内嵌" 或 "上传"
if DATA_MODE == "上传":
    # 在Colab左侧文件面板上传这两个文件，路径必须与下面一致。
    terms_path = Path("/content/glossary_CN.json")
    dataset_path = Path("/content/glossary_sft_CN.jsonl")
    if not terms_path.is_file() or not dataset_path.is_file():
        raise FileNotFoundError("请先将 glossary_CN.json 和 glossary_sft_CN.jsonl 上传到 /content/")
    TERMS = json.loads(terms_path.read_text(encoding="utf-8"))["terms"]
    RAW = [json.loads(line) for line in dataset_path.read_text(encoding="utf-8").splitlines() if line.strip()]
elif DATA_MODE != "内嵌":
    raise ValueError("DATA_MODE 必须是‘内嵌’或‘上传’")
# 本实验按词条顺序每两行划分训练和验证，因此要求文件与词汇表逐条一致。
assert len(RAW) == 2 * len(TERMS), "问答条数应为词条数的两倍，请检查文件版本"
for i, term in enumerate(TERMS):
    expected_questions = [f"“{term['term']}”是什么意思？", f"请简要解释{term['term']}。"]
    for offset, question in enumerate(expected_questions):
        msgs = RAW[2 * i + offset].get("messages", [])
        assert len(msgs) == 2 and msgs[0].get("role") == "user" and msgs[1].get("role") == "assistant", "消息格式不符"
        assert msgs[0].get("content") == question and msgs[1].get("content") == term["definition"], "词汇表与JSONL内容或顺序不一致"
print("词汇表数据来源：", DATA_MODE, "；词条数：", len(TERMS), "；问答数：", len(RAW))

BY_TERM={t['term']:t for t in TERMS}
BY_ID={t['id']:t for t in TERMS}
assert len(RAW)==2*len(TERMS)
gloss_train=[]; gloss_valid=[]
# 按原脚本两种问法拆分；两组词条相同，但问题文字不同。
for i,record in enumerate(RAW):
    q=record['messages'][0]['content']; answer=record['messages'][1]['content']
    ex=example(q,answer,False,'词汇表内','glossary','',[])
    (gloss_train if i%2==0 else gloss_valid).append(ex)
for bucket,term in [(gloss_train,'卡尔曼滤波'),(gloss_train,'二叉堆'),(gloss_valid,'泊松分布')]:
    bucket.append(example(term+'是什么？','词汇表没有记载，需要确认。',True,'词汇表外','glossary','',['确认']))
# 自编少量测试和gold：术语、改写、表外各两题；训练中出现过词条，但未出现测试问句。
TESTS=[('lora','低秩适应是什么意思？','直接',['权重','矩阵']),('token','请说明词元的含义。','直接',['单位']),('lora','固定基础权重并只训练小矩阵的方法是什么？','改写',['权重','矩阵']),('token','语言模型计算文字长度常用的单位是什么？','改写',['单位'])]
gloss_test=[example(q,BY_ID[id]['definition'],False,g,'glossary','',kw) for id,q,g,kw in TESTS]
for q in ['课程词汇表中傅里叶变换是什么意思？','课程词汇表中二分搜索是什么意思？']:
    gloss_test.append(example(q,'词汇表没有记载，需要确认。',True,'表外','glossary','',['确认']))
assert not ({e['question'] for e in gloss_train}&{e['question'] for e in gloss_test})
gold={e['question']:{'needs_review':e['needs_review'],'keywords':e['keywords'],'answer':e['answer']} for e in gloss_test}
(WORK/'glossary_test_gold.json').write_text(json.dumps(gold,ensure_ascii=False,indent=2),encoding='utf-8')
print('词汇表训练/验证/测试：',len(gloss_train),len(gloss_valid),len(gloss_test)); display(pd.DataFrame(gloss_test)[['group','question','answer','needs_review']])

def retrieve(ex):
    """只用问题中的词条名和别名检索，不读取标准答案；未命中时请求确认。"""
    q=ex['question'].casefold(); candidates=[]
    for term in TERMS:
        matches=[name for name in [term['term'],*term['aliases']] if len(name)>=2 and name.casefold() in q]
        if matches: candidates.append((max(len(x) for x in matches),term))
    candidates.sort(key=lambda v:v[0],reverse=True)
    if not candidates or (len(candidates)>1 and candidates[0][0]==candidates[1][0]): return json.dumps({'answer':'未检索到唯一词条，需要确认。','needs_review':True},ensure_ascii=False)
    return json.dumps({'answer':candidates[0][1]['definition'],'needs_review':False},ensure_ascii=False)

In [ ]:
# 【中文语法说明】每次赋值把评价汇总或训练结果保存到变量，之后才能汇总比较。
# answer_fn=retrieve 将检索函数作为参数传入评价函数；这里只比较直接检索，不是生成模型。
# 三次评价使用同一词汇表测试集；训练后与训练前之间发生了适配器训练。
gloss_base=load_base()
gloss_before=evaluate(gloss_base,gloss_test,'glossary_before')
retrieval=evaluate(None,gloss_test,'glossary_retrieval',answer_fn=retrieve)
gloss_tuned,gloss_trainer,gloss_seconds=train_lora(gloss_base,gloss_train,gloss_valid,'glossary',CONFIG['epochs_glossary'])
gloss_after=evaluate(gloss_tuned,gloss_test,'glossary_after')
display(pd.DataFrame([gloss_before,gloss_after,retrieval]))
print('注意：检索输出的JSON由程序构造，不能当作模型格式能力；表外不代表模型原本不知道。')

## 10. 两组结果分别查看
FAQ结果与词汇表结果使用不同题目，分数不能直接作为相同难度排名；检索JSON由程序构造，其格式分数不表示模型能力。

In [ ]:
# 【中文语法说明】嵌套字典用分层的键组织FAQ与词汇表结果。
# ensure_ascii=False 保留中文，indent=2 使输出易读；write_text 将文本写入指定文件。
# executed_training_myself 只能在自己实际完成两次训练后设置为True。
summary={"faq":faq_original_result,"glossary":{"before":gloss_before,"after":gloss_after,"retrieval":retrieval,"训练时间秒":gloss_seconds},"executed_training_myself":True}
(WORK/"result_both.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
display(pd.DataFrame([gloss_before,gloss_after,retrieval]))
print("FAQ结果在 /content/qlora_lab；词汇表结果在",WORK)
